# Notebook 06: ネットワークセキュリティの基礎

Web セキュリティを学ぶには、ネットワークの仕組みの理解が欠かせません。
データがどう流れ、どこで盗聴・改ざんされる可能性があるかを学びます。

## 学習目標
1. TCP/IP とポートの基本を理解する
2. パケットキャプチャの概念を知る
3. TLS/SSL による暗号化通信を理解する
4. DNS の仕組みと攻撃を学ぶ

---

**ネットワークスキャンは自分の管理するネットワークのみで行ってください**

## 1. TCP/IP の基本

インターネット通信は TCP/IP プロトコルスタックで行われます。

```
アプリケーション層  HTTP, HTTPS, DNS, SSH
トランスポート層    TCP, UDP
インターネット層    IP (IPv4, IPv6)
リンク層           Ethernet, Wi-Fi
```

### IP アドレス
- ネットワーク上のコンピューターの住所
- IPv4: `192.168.1.1`（32ビット）
- IPv6: `2001:db8::1`（128ビット）
- `127.0.0.1` = ローカルホスト（自分自身）

### ポート番号
- コンピューター内のサービスの窓口番号（0-65535）
- よく使うポート:

| ポート | サービス | 説明 |
|--------|----------|------|
| 80     | HTTP     | Web（暗号化なし） |
| 443    | HTTPS    | Web（暗号化あり） |
| 22     | SSH      | リモートログイン |
| 53     | DNS      | 名前解決 |
| 3306   | MySQL    | データベース |
| 5432   | PostgreSQL | データベース |

In [ ]:
import socket

# === 自分のネットワーク情報 ===
print("=== ホスト情報 ===")
hostname = socket.gethostname()
print(f"ホスト名: {hostname}")

try:
    local_ip = socket.gethostbyname(hostname)
    print(f"ローカルIP: {local_ip}")
except socket.gaierror:
    print("ローカルIPの取得に失敗")

print(f"ループバック: 127.0.0.1")
print()

# === DNS ルックアップ ===
print("=== DNS ルックアップ ===")
domains = ["example.com", "google.com", "github.com"]
for domain in domains:
    try:
        ip = socket.gethostbyname(domain)
        print(f"{domain} → {ip}")
    except socket.gaierror as e:
        print(f"{domain} → エラー: {e}")

In [ ]:
import socket

# === ポートスキャンのシミュレーション ===
# ※ 自分のマシン（localhost）のみスキャンする

print("=== localhost のポートスキャン ===")
print("※ 自分のマシンのみを対象にしています")
print()

common_ports = {
    22: "SSH",
    80: "HTTP",
    443: "HTTPS",
    3000: "dev server",
    3306: "MySQL",
    5000: "Flask",
    5432: "PostgreSQL",
    8080: "HTTP alt",
    8888: "Jupyter",
}

for port, service in common_ports.items():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(0.5)
    result = sock.connect_ex(('127.0.0.1', port))
    status = "OPEN" if result == 0 else "closed"
    if result == 0:
        print(f"  ポート {port:5d} ({service:12s}): {status}")
    sock.close()

print()
print("実際のペネトレーションテストでは nmap などの専用ツールを使います")
print("例: nmap -sV 192.168.1.0/24")

## 2. パケットと盗聴

ネットワーク通信はパケット（小さなデータの塊）で行われます。

### HTTP（暗号化なし）の危険性
```
あなた  ──── [パスワード: abc123] ────→  サーバー
            ↑
       攻撃者がパケットを盗聴
       パスワードが丸見え！
```

### HTTPS（暗号化あり）
```
あなた  ──── [暗号化されたデータ] ────→  サーバー
            ↑
       攻撃者がパケットを盗聴
       中身が読めない！
```

### 盗聴が可能な場所
- 公衆 Wi-Fi（カフェ、空港など）
- 同一ネットワーク上の攻撃者
- ISP（インターネットプロバイダー）
- 中間者（Man-in-the-Middle）攻撃

In [ ]:
# HTTP vs HTTPS のデータの違いをシミュレーション

import json

# ログインリクエストのデータ
login_data = {
    "username": "alice",
    "password": "super_secret_password",
    "session_token": "abc123def456"
}

print("=== HTTP（暗号化なし）で盗聴した場合 ===")
print("攻撃者が見える内容:")
print(f"POST /login HTTP/1.1")
print(f"Host: example.com")
print(f"Content-Type: application/json")
print(f"")
print(json.dumps(login_data, indent=2))
print("→ 全て丸見え！")
print()

print("=== HTTPS（TLS暗号化）で盗聴した場合 ===")
print("攻撃者が見える内容:")
# TLS暗号化後のデータ（ダミー）
import os
encrypted = os.urandom(len(json.dumps(login_data)))
print(f"17 03 03 {' '.join(f'{b:02x}' for b in encrypted[:32])}")
print(f"...（暗号化されたバイナリデータ）")
print("→ 中身が分からない！")

## 3. TLS/SSL の仕組み

TLS（Transport Layer Security）は通信を暗号化するプロトコルです。
HTTPS = HTTP + TLS です。

### TLS ハンドシェイク（簡略版）
```
クライアント                         サーバー
    |                                  |
    | ─── ClientHello ───────────────→ |  (対応する暗号方式の提示)
    |                                  |
    | ←── ServerHello + 証明書 ────── |  (証明書で身元を証明)
    |                                  |
    | ─── 鍵交換 ───────────────────→ |  (共通鍵を安全に共有)
    |                                  |
    | ←─────── 暗号化通信 ──────────→ |  (以降は全て暗号化)
```

### 証明書の役割
- サーバーの身元を証明する（認証局 CA が発行）
- ブラウザの鍵マークが証明書の確認結果

In [ ]:
import ssl
import socket

# === サーバー証明書の確認 ===
print("=== サーバー証明書の確認 ===")

def check_certificate(hostname: str, port: int = 443):
    """サーバーのTLS証明書を確認する"""
    context = ssl.create_default_context()
    try:
        with socket.create_connection((hostname, port), timeout=5) as sock:
            with context.wrap_socket(sock, server_hostname=hostname) as ssock:
                cert = ssock.getpeercert()
                print(f"  ホスト: {hostname}")
                print(f"  TLSバージョン: {ssock.version()}")
                print(f"  暗号方式: {ssock.cipher()[0]}")
                # 証明書の発行者
                issuer = dict(x[0] for x in cert['issuer'])
                print(f"  発行者: {issuer.get('organizationName', 'N/A')}")
                # 有効期限
                print(f"  有効期限: {cert['notBefore']} ～ {cert['notAfter']}")
                print()
    except Exception as e:
        print(f"  {hostname}: 接続エラー ({e})")
        print()

check_certificate("example.com")
check_certificate("github.com")

## 4. DNS とその攻撃

DNS（Domain Name System）はドメイン名を IP アドレスに変換するシステムです。

```
ブラウザ: "example.com にアクセスしたい"
    ↓
DNS サーバー: "example.com は 93.184.216.34 です"
    ↓
ブラウザ: 93.184.216.34 に接続
```

### DNS の攻撃
- **DNS スプーフィング**: 偽の IP アドレスを返して偽サイトに誘導
- **DNS キャッシュポイズニング**: DNS サーバーのキャッシュを汚染
- **DNS ハイジャック**: DNS の設定を書き換える

### 防御
- **DNSSEC**: DNS 応答に署名を付けて改ざんを検出
- **DNS over HTTPS (DoH)**: DNS 通信を暗号化
- 信頼できる DNS サーバーの使用（例: 1.1.1.1, 8.8.8.8）

In [ ]:
import socket

# === DNS ルックアップの詳細 ===
print("=== DNS ルックアップの詳細 ===")

def dns_lookup(domain: str):
    """ドメインのDNS情報を取得"""
    print(f"\n{domain}:")
    try:
        # 全てのアドレス情報を取得
        results = socket.getaddrinfo(domain, None)
        seen = set()
        for family, _, _, _, addr in results:
            ip = addr[0]
            if ip not in seen:
                seen.add(ip)
                family_name = "IPv4" if family == socket.AF_INET else "IPv6"
                print(f"  {family_name}: {ip}")
    except socket.gaierror as e:
        print(f"  エラー: {e}")

for domain in ["example.com", "google.com", "github.com"]:
    dns_lookup(domain)

print()
print("=== 逆引き DNS ===")
try:
    host = socket.gethostbyaddr("8.8.8.8")
    print(f"8.8.8.8 → {host[0]}")
except socket.herror as e:
    print(f"8.8.8.8 → 逆引き失敗: {e}")

## まとめ

| トピック | 脅威 | 防御 |
|----------|------|------|
| 通信の盗聴 | HTTP での平文通信 | HTTPS (TLS) の使用 |
| ポートスキャン | 不要なサービスの露出 | ファイアウォール・不要ポートの閉鎖 |
| DNS | スプーフィング・キャッシュ汚染 | DNSSEC・DoH |
| 中間者攻撃 | 通信の傍受・改ざん | TLS 証明書の検証 |

## 演習

1. `socket.gethostbyname()` を使って自分がよくアクセスするサイトの IP アドレスを調べてみよう
2. localhost のポートスキャンを実行して、どのサービスが動いているか確認してみよう
3. `ssl` モジュールで好きなサイトの TLS 証明書を確認してみよう
4. HTTP と HTTPS でアクセスした場合のブラウザの表示の違いを確認してみよう